## 🟢 개와고양이 사전학습 2  


- 1. 투스테이지(개와고양이_사전학습1.ipynb)  
    - CNN동결 VGG의 특징을 미리 계산하고 numpy배열로 바꾼다..  
    

- 2. 인라인(개와고양이_사전학습2.ipynb)  
    - ㅇ  
    - ㅇ  
    - 사람들이 많이 쓰는 방법이다.  


In [ ]:
# VGG19 사전학습 모델 사용하기
import os, shutil, pathlib
import tensorflow as tf
import keras
import pickle
from keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import random

original_dir = pathlib.Path("./dogs-vs-cats/train")
new_base_dir = pathlib.Path("./dogs-vs-cats/dogs-vs-cats_small")


# 폴더로 옮기기 => 데이터셋은 폴더를 지정하면 자동으로 라벨링을 한다. 폴더이름을 오름차순으로 정렬해서 자동 라벨링
# 데이터셋 - train -  cats
#                   dogs
#          test  -  cats
#                   dogs
#       validation -  cats
#                   dogs


# 폴더 지정, 시작 인덱스, 종료 인덱스
def make_subset(subset_name, start_index, end_index):  # make_subset("train", 0, 1000)
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir, exist_ok=True)  # 디렉토리가 없을 경우 새로 디렉토리를 만들어라
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)


make_subset("train", 0, 1000)
make_subset("validation", 1000, 1500)
make_subset("test", 1500, 2000)

from keras.utils import image_dataset_from_directory

# batch_size에 지정된 만큼 폴더로부터 이미지를 읽어온다. 크기는 image_size에 지정한 값으로 가져온다
train_ds = image_dataset_from_directory(
    new_base_dir / "train", image_size=(180, 180), batch_size=16
)
validation_ds = image_dataset_from_directory(
    new_base_dir / "validation", image_size=(180, 180), batch_size=16
)
test_ds = image_dataset_from_directory(
    new_base_dir / "test", image_size=(180, 180), batch_size=16
)

# VGG19 이미지 모델 가져오기
from keras.applications.vgg19 import VGG19

conv_base = keras.applications.vgg19.VGG19(
    weights="imagenet",
    include_top=False,  # CNN만 가져와라 , CNN이 하단에 있음, 상단-완전연결망(분류)
    input_shape=(180, 180, 3),  # 입력할 데이터 크기를 주어야 한다
    # 데이터셋에서 지정한 크기와 일치해야 한다
)

conv_base.trainable = True
print(
    f"합성곱 기반 층을 동결 후의 훈련 가능한 가중치 개수 : {len(conv_base.trainable_weights)}"
)
conv_base.trainable = False
print(
    f"합성곱 기반 층을 동결 후의 훈련 가능한 가중치 개수 : {len(conv_base.trainable_weights)}"
)

# 데이터 증강
data_argumentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.4),
    ]
)

# 모델 만들기
input = keras.Input(shape=(180, 180, 3))  # 모델의 입력레이어 정의
x = data_argumentation(input)  # 입력이미지에 데이터 증강을 적요한다.
x = keras.applications.vgg19.preprocess_input(x)  # VGG19에 맞는 전처리 작업


history = model.fit(
    train_ds, epochs=10, validation_data=validation_ds, callbacks=callbacks
)

with open("개와고앙이_사전학습_2.bin", "wb") as file:
    pickle.dump(history.history, file)